In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 7 — O CORTE DE OCCAM: FEATURE SELECTION

> "Entidades não devem ser multiplicadas além da necessidade. A explicação mais simples é provavelmente a mais acurada."
> — Guilherme de Occam

A engenharia foi produtiva de um jeito inesperado: ela me ensinou que eu já tinha *features* demais, não de menos. Olhando a tabela com trinta colunas — muitas ecoando umas às outras —, senti que estava me afogando em informação redundante.

Lembrei da Navalha de Occam. Em *machine learning*, ela se traduz num princípio poderoso: um modelo mais simples tende a generalizar melhor, é mais fácil de interpretar e menos propenso a *overfitting*. No OncoClassify 1.0 usei tudo sem questionar, na crença ingênua de que "mais dados" significa "melhor modelo". Desta vez, eu seria um curador: pegaria a navalha e cortaria o excesso, para que o verdadeiro sinal brilhasse.

## Feature Selection: Por Que Menos é Mais

Seleção de *features* é escolher um subconjunto relevante das variáveis existentes. Reduz *overfitting*, acelera o treino e melhora a interpretabilidade. Diferente da engenharia (que *cria*), aqui nós *escolhemos*. Há três famílias:

**Filtro** — avaliam cada *feature* isoladamente por um teste estatístico (rápidos; ignoram interações). Ex.: Informação Mútua.

**Wrapper** — treinam o modelo repetidamente com diferentes subconjuntos (poderosos; caros). Ex.: RFE.

**Embutido** — a seleção faz parte do próprio treino. Ex.: importância de *features* do Random Forest.

Vamos aplicar um de cada tipo dos dois primeiros grupos — Informação Mútua (filtro) e importância do Random Forest (embutido) — sobre o conjunto **completo** (30 originais + as 3 da engenharia), para testar de uma vez quais *features* dominam e confirmar se as criadas no capítulo anterior merecem ficar.

#### Código 7.1: Método de Filtro — Informação Mútua


In [ ]:
import pandas as pd
from sklearn.feature_selection import mutual_info_classif
from oncoclassify_utils import X_train, y_train
# reaproveitamos a funcao do Capitulo 6
from oncoclassify_utils import FEATURES_MORFOLOGICAS

def adicionar_features_derivadas(X):
    X = X.copy()
    X["ratio_raio_textura"]     = X["mean radius"] / X["mean texture"]
    X["irregularidade_forma"]   = (X["mean perimeter"] ** 2) / (X["mean area"] + 1e-6)
    X["assimetria_concavidade"] = X["worst concavity"] / (X["worst symmetry"] + 1e-6)
    return X

X_sel = adicionar_features_derivadas(X_train)       # 33 colunas (30 + 3)
mi = mutual_info_classif(X_sel, y_train, random_state=42)
mi_scores = pd.Series(mi, index=X_sel.columns).sort_values(ascending=False)
print("--- Top 10 por Informação Mútua ---")
print(mi_scores.head(10).round(4).to_string())

--- Top 10 por Informação Mútua ---
worst perimeter         0.4682
mean concave points     0.4595
worst radius            0.4581
worst area              0.4557
worst concave points    0.4481
mean perimeter          0.4081
mean concavity          0.3686
mean area               0.3600
mean radius             0.3547
area error              0.3463


![Importância por Informação Mútua](media/figura_07_1.png)

O topo é dominado pelas medidas "worst" (o pior estado do tumor) e por `concave points` — confirmando a intuição da EDA. Nenhuma das três *features* de engenharia aparece no Top 10.

#### Código 7.2: Método Embutido — Importância do Random Forest


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_sel, y_train)
rf_imp = pd.Series(rf.feature_importances_, index=X_sel.columns).sort_values(ascending=False)
print("--- Top 10 por Importância do Random Forest ---")
print(rf_imp.head(10).round(4).to_string())

# Onde ficaram as features de engenharia, num ranking de 33?
for c in ["ratio_raio_textura", "irregularidade_forma", "assimetria_concavidade"]:
    print(f"posicao de {c}: #{list(mi_scores.index).index(c) + 1} (Info. Mútua)")

--- Top 10 por Importância do Random Forest ---
worst radius            0.1237
worst concave points    0.1049
mean concave points     0.1020
worst perimeter         0.0977
worst area              0.0931
mean concavity          0.0630
mean radius             0.0603
mean perimeter          0.0515
worst concavity         0.0437
area error              0.0333
posicao de ratio_raio_textura: #27 (Info. Mútua)
posicao de irregularidade_forma: #18 (Info. Mútua)
posicao de assimetria_concavidade: #12 (Info. Mútua)


![Importância por Random Forest](media/figura_07_2.png)

Dois métodos independentes concordam de forma notável: `worst radius`, `worst concave points`, `mean concave points`, `worst perimeter` e `worst area` são o coração do sinal. Essa convergência dá confiança de que essas *features* são realmente as decisivas.

E as três *features* de engenharia? Ficaram nas posições **#12, #18 e #27** de 33. A mais forte, `assimetria_concavidade`, chega a #12 — respeitável, mas ainda abaixo das originais que a "contêm". Isso fecha, com número, a decisão do capítulo anterior.

> **📘 DECISÃO DE PROJETO — O conjunto de modelagem**
>
> A partir daqui, a base de modelagem do livro são as **30 features morfológicas** curadas. As três *features* de engenharia foram criadas, testadas e honestamente **postas de lado** por redundância (Capítulo 6). A seleção mostrou ainda que poderíamos reduzir a ~10–15 *features* sem perda relevante — e é exatamente isso que faremos dentro de um *pipeline* no Capítulo 13. Menos, porém melhor: a navalha de Occam em ação.

> **📌 CHECKLIST DO CAPÍTULO 7**
>
> ✅ Entende os benefícios da seleção: menos *overfitting*, mais velocidade e interpretabilidade.
>
> ✅ Conhece as três famílias: filtro, wrapper e embutido.
>
> ✅ Aplicou Informação Mútua e importância do Random Forest, observando forte concordância.
>
> ✅ Confirmou, por ranking, que as *features* de engenharia são redundantes (posições #12/#18/#27).
>
> **⚠️ CRÍTICO:** Seleção, como engenharia, é feita **apenas no treino**. Selecionar *features* usando o dataset completo é *data leakage*.

A navalha de Occam fez seu trabalho. A névoa de trinta e três variáveis se dissipou, revelando um núcleo duro de preditores poderosos e um punhado de redundâncias. Eu não tinha mais uma lista: tinha uma hierarquia. Sabia quais *features* eram os generais e quais eram apenas soldados.

Com o campo de batalha mapeado, esculpido e podado, a fase de preparação estava completa. Era hora de escolher as armas — o arsenal de algoritmos de classificação. E eu começaria pelo mais antigo e elegante de todos, baseado na pura lógica da probabilidade: o Teorema de Bayes.

**PARTE III — O ARSENAL DE ALGORITMOS**
